In [6]:
!ollama pull gemma:2b

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest 
pulling c1864a5eb193:   0% ▕                  ▏ 4.9 MB/1.7 GB                  pulling manifest 
pulling c1864a5eb193:   0% ▕                  ▏ 5.6 MB/1.7 GB                  pulling manifest 
pulling c1864a5eb193:   1% ▕                  ▏  11 MB/1.7 GB                  pulling manifest 
pulling c1864a5eb193:   1% ▕                  ▏  15 MB/1.7 GB                  pulling manifest 
pulling c1864a5eb193:   1% ▕                  ▏  18 MB/1.7 GB                  pulling manifest 
pulling c1864a5eb193: 

In [ ]:
import os
import io
import base64
from typing import Optional
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.middleware.cors import CORSMiddleware
from dotenv import load_dotenv
import ollama
from openai import OpenAI
import asyncio
import uvicorn
import nest_asyncio

nest_asyncio.apply()

# 환경 변수 로드
load_dotenv()

app = FastAPI()

# CORS 설정 (가이드 준수: 모든 허용)
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

def getModelResponse(imageContent: bytes, userQuestion: str):
    """ 
    설정된 모델(GPT 또는 OLLAMA)에 따라 이미지 분석 결과를 반환하는 함수입니다.
    """
    try:
        useModel = os.getenv("USE_MODEL", "OLLAMA")
        
        if useModel == "GPT":
            # OpenAI GPT-4o Vision 활용 (텍스트 추출 및 질문 처리)
            client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
            base64Image = base64.b64encode(imageContent).decode('utf-8')
            
            response = client.chat.completions.create(
                model="gpt-4o",
                messages=[
                    {
                        "role": "user",
                        "content": [
                            {"type": "text", "text": userQuestion},
                            {
                                "type": "image_url",
                                "image_url": {
                                    "url": f"data:image/jpeg;base64,{base64Image}",
                                },
                            },
                        ],
                    }
                ],
            )
            return response.choices[0].message.content
            
        elif useModel == "OLLAMA":
            # 로컬 Ollama gemma4:e2b 모델 활용
            ollamaModel = os.getenv("OLLAMA_MODEL", "gemma4:e2b")
            response = ollama.chat(
                model=ollamaModel,
                messages=[{
                    'role': 'user',
                    'content': userQuestion,
                    'images': [imageContent]
                }]
            )
            return response['message']['content']
            
        else:
            return "지원하지 않는 모델 설정입니다."
            
    except Exception as e:
        raise e

@app.post("/analyze")
async def analyzeImage(image: UploadFile = File(...), question: str = Form(...)):
    """ 
    사용자가 업로드한 이미지와 질문을 받아 분석 결과를 JSON으로 반환하는 API 엔드포인트입니다.
    """
    try:
        # 이미지 데이터 읽기
        imageContent = await image.read()
        
        # 모델을 통한 분석 수행
        analysisResult = getModelResponse(imageContent, question)
        
        # 데이터 처리 예시 (가이드 준수: 리스트 컴프리헨션 금지 및 명시적 반복문 사용)
        wordList = analysisResult.split()
        processedWords = []
        for i in range(0, len(wordList)):
            processedWords.append(wordList[i])
            
        return {
            "success": True,
            "result": analysisResult,
            "metadata": {
                "fileName": image.filename,
                "wordCount": len(processedWords)
            }
        }
        
    except Exception as e:
        # 에러 발생 시 가이드에 정의된 JSON 형식 반환
        return {
            "success": False,
            "message": str(e)
        }

if __name__ == "__main__":
    import nest_asyncio
    nest_asyncio.apply()
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
    server = uvicorn.Server(config)

    # 실행 전에 주소를 미리 출력합니다
    print("서버가 성공적으로 실행되었습니다!")
    print("로컬 주소: http://127.0.0.1:8000")
    print("외부 접속: http://0.0.0.0:8000")
    print("API 문서: http://127.0.0.1:8000/docs")

    await server.serve()

INFO:     Started server process [28868]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


서버가 성공적으로 실행되었습니다!
로컬 주소: http://127.0.0.1:8000
외부 접속: http://0.0.0.0:8000
API 문서: http://127.0.0.1:8000/docs
INFO:     127.0.0.1:54680 - "POST /analyze HTTP/1.1" 200 OK
